# Rendementsscan thuisbatterij
Effect van het round-trip rendement (40-90%) op de ideale accugrootte, voor batterijen van
5 tot 30 kWh. Zelfde simulatie als V2: laden uit teruglevering, ontladen alleen in uren zonder
teruglevering, 2,5 kW vermogen. De ideale grootte per dag is begrensd door de capaciteit.

In [ ]:
import sqlite3, os, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ACCU_KW = 2.5
RENDEMENTEN = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]
CAPACITEITEN = [5, 10, 15, 20, 25, 30]
DREMPEL = 0.99
DAGEN = 400
GEEN_TERUG = 0.01
DB = next(p for p in ["/config/home-assistant_v2.db", "/homeassistant/home-assistant_v2.db"] if os.path.exists(p))
IMPORT_IDS = ["sensor.slimmemeter_energy_delivered_tariff_1", "sensor.slimmemeter_energy_delivered_tariff_2"]
EXPORT_IDS = ["sensor.slimmemeter_energy_returned_tariff_1", "sensor.slimmemeter_energy_returned_tariff_2"]
con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)

def reeks(sid):
    mid, unit = con.execute("SELECT id, unit_of_measurement FROM statistics_meta WHERE statistic_id=?", (sid,)).fetchone()
    df = pd.read_sql("SELECT start_ts, sum FROM statistics WHERE metadata_id=? AND start_ts>=? ORDER BY start_ts",
                     con, params=(mid, time.time() - DAGEN * 86400))
    idx = pd.DatetimeIndex(pd.to_datetime(df.start_ts, unit="s", utc=True)).tz_convert("Europe/Amsterdam")
    s = pd.Series(df["sum"].values, index=idx).diff().clip(lower=0)
    return s / 1000 if unit == "Wh" else s

def som(ids):
    out = None
    for sid in ids:
        out = reeks(sid) if out is None else out.add(reeks(sid), fill_value=0)
    return out

uur = pd.DataFrame({"afname": som(IMPORT_IDS), "teruglevering": som(EXPORT_IDS)}).dropna().asfreq("h", fill_value=0)
uur["pv_uit"] = uur.teruglevering < GEEN_TERUG
IMP, EXP, UIT = uur.afname.values, uur.teruglevering.values, uur.pv_uit.values
print(f"{uur.index.min():%d-%m-%Y} t/m {uur.index.max():%d-%m-%Y}, {len(uur)} uren")

In [ ]:
def geleverd(imp, exp, uit, cap, eta, kw=ACCU_KW):
    eff = eta ** 0.5
    soc = tot = 0.0
    for i, e, u in zip(imp, exp, uit):
        if cap > soc:
            soc += min(e, kw, (cap - soc) / eff) * eff
        if u:
            ont = min(i, kw, soc * eff)
            soc -= ont / eff
            tot += ont
    return tot

def ideaal(imp, exp, uit, eta):
    vol = geleverd(imp, exp, uit, 1e6, eta)
    if vol < 0.05:
        return 0.0
    lo, hi = 0.0, 60.0
    for _ in range(22):
        mid = (lo + hi) / 2
        lo, hi = (lo, mid) if geleverd(imp, exp, uit, mid, eta) >= DREMPEL * vol else (mid, hi)
    return hi

# vensters per dag: eerste terugleveruur tot eerste terugleveruur volgende dag
dagen = sorted(set(uur.index.normalize()))
start = {}
for d in dagen:
    t = uur.loc[d:d + pd.Timedelta(hours=23)]
    t = t.index[~t.pv_uit]
    start[d] = t[0] if len(t) else None
vensters = []
for k, d in enumerate(dagen[:-1]):
    if start[d] is None:
        vensters.append(None); continue
    e = start[dagen[k + 1]] or dagen[k + 1] + pd.Timedelta(hours=12)
    w = uur.loc[start[d]:e - pd.Timedelta(hours=1)]
    vensters.append((w.afname.values, w.teruglevering.values, w.pv_uit.values))

rijen = []
for eta in RENDEMENTEN:
    ideaal_dag = np.array([ideaal(*v, eta) if v else 0.0 for v in vensters])
    for cap in CAPACITEITEN:
        x = np.minimum(ideaal_dag, cap)
        jaar = geleverd(IMP, EXP, UIT, cap, eta) * 365 / len(vensters)
        rijen.append(dict(rendement=eta, capaciteit=cap, mediaan=np.median(x), gemiddeld=x.mean(),
                          p80=np.percentile(x, 80), p90=np.percentile(x, 90),
                          dagen_vol=int((ideaal_dag >= cap).sum()), geleverd_kwh_jaar=jaar))
res = pd.DataFrame(rijen)
res.round(2).to_csv("rendement_scan.csv", sep=";", decimal=",", index=False)
res.round(1)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
kleuren = plt.cm.viridis(np.linspace(0, 0.9, len(CAPACITEITEN)))
for c, kl in zip(CAPACITEITEN, kleuren):
    r = res[res.capaciteit == c]
    axs[0].scatter(r.rendement * 100, r.p80, color=kl, s=40, label=f"{c} kWh")
    axs[0].plot(r.rendement * 100, r.p80, color=kl, lw=1, alpha=.6)
    axs[1].scatter(r.rendement * 100, r.geleverd_kwh_jaar, color=kl, s=40, label=f"{c} kWh")
    axs[1].plot(r.rendement * 100, r.geleverd_kwh_jaar, color=kl, lw=1, alpha=.6)
axs[0].set(xlabel="rendement accu (%)", ylabel="ideale accugrootte, P80 (kWh)", title="Ideale accugrootte (80% van de dagen)")
axs[1].set(xlabel="rendement accu (%)", ylabel="geleverd uit accu (kWh/jaar)", title="Jaaropbrengst")
axs[0].legend(title="capaciteit"); plt.tight_layout(); plt.show()